# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook guides you through loading and exploring the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keep as the object, not a dict
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below we display the available record sets and their fields for inspection. All identifiers are referenced by their `@id` as per Croissant conventions.


In [ ]:
# Show the record sets in this dataset, with their @id
print("Available record sets (@id):")
for rset in metadata.record_sets:
    print(f"  - {rset.id} : {rset.name}")
    print("    Fields:")
    for field in rset.fields:
        print(f"        * {field.id} : {field.name} (type={field.data_type})")

Below we display an example record from each record set. This helps verify how to access the individual entries.

> **Note**: Replace `<record_set_id>` with the actual `@id` of a record set of interest in further analysis.

In [ ]:
# Print an example record from each record set
for rset in metadata.record_sets:
    record_set_id = rset.id
    try:
        example_records = list(dataset.records(record_set=record_set_id))
        if example_records:
            print(f"First record in '{record_set_id}':\n{json.dumps(example_records[0], indent=2)}\n")
        else:
            print(f"No records found in '{record_set_id}'.\n")
    except Exception as e:
        print(f"Error loading records for record set '{record_set_id}': {e}\n")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis.

We demonstrate this using the primary survey responses record set. 

Please choose the appropriate `@id` of the record set to analyze (from above --- in this dataset, it's typically similar to `'cr:RecordSet:main'`, but always check!). We'll extract all records from this primary record set and convert them to a DataFrame.

In [ ]:
# Collect all record set IDs, pick the main one for this example
record_set_ids = [rset.id for rset in metadata.record_sets]
print("All detected record sets:", record_set_ids)

# Try to select the main record set automatically
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # If only one or if the main is first; change if needed
    print(f"Using record set: {main_record_set_id}")
else:
    raise ValueError("No record sets detected in metadata.")

# Extract data from all record sets into DataFrames
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

# Show example columns and head of the main record set
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Below we:
- Select a numeric field for basic filtering and normalization
- Filter the records to those above a numeric threshold
- Normalize the field and review the result
- Group by a categorical field

**Using entity `@id`s for all field/column references.**

First, display all columns so the user can pick appropriate fields for further EDA.


In [ ]:
# List available fields/columns for EDA
df = dataframes[main_record_set_id]
print("Available columns:")
print(df.columns.tolist())

For this EDA, we select the following fields (by their column `@id`):
- Numeric field: `'psq_total_score'` (assumed field measuring perceived stress, as per GAD-7/PHQ-9/PSQ columns listed in the data description)
- Group field: `'sex'` (demographic grouping)

> **Change these `field_id`s as needed based on the columns listed above if your data has different precise `@id`s.**

In [ ]:
# Select field identifiers (columns) by their @id/name
numeric_field_id = 'psq_total_score'   # Replace with actual id as shown in the column list above if different
group_field_id = 'sex'  # For grouping by gender, replace with correct id as needed

# Confirm both fields exist, adjust if necessary
if numeric_field_id not in df.columns:
    raise ValueError(f"Numeric field '{numeric_field_id}' not found in columns: {df.columns.tolist()}")

if group_field_id not in df.columns:
    print(f"Group field '{group_field_id}' not found; skipping grouping step.")

# Convert to numeric, handle possible missing/invalid data
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
)
filtered_df[f"{numeric_field_id}_normalized"] /= filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field if available
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the data.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(7,4))
df[numeric_field_id].dropna().hist(bins=20)
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Box plot grouped by group_field_id if present
if group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant` directly from its FAIR schema
- Inspected the available record sets and their fields (by `@id`)
- Extracted and explored survey data in a pandas DataFrame
- Performed basic EDA: filtering, normalization, and grouping
- Visualized data distributions by field and by demographic group

This workflow demonstrates a reproducible pipeline for AI-ready data using machine-readable schema standards, supporting further statistical or machine learning analysis.
